# Laboratorio Modelos de Clasificación — Ejercicio 1
## Santander Customer Satisfaction

**Plantilla de trabajo grupal para Google Colab** · Equipo: **Poma, Aldana, Brickzon, Vera y Marco**.

**Finalidad:** extender el ejercicio del docente con **4 escaladores, 5 clasificadores nuevos y 5 modelos híbridos**, implementar una comparación reproducible y escribir conclusiones. **Las celdas de código de la Parte B se dejan intencionalmente vacías.** Cada integrante debe leer la celda de instrucciones inmediatamente anterior y escribir allí su implementación.

> **Orden de lectura:** normas de trabajo → **Parte A: referencia original** → **Parte B: desarrollo grupal**. La Parte A conserva las celdas originales del notebook Santander, sin sus salidas; **no sustituye la evaluación metodológicamente corregida de la Parte B**. Para la entrega, ejecutar la Parte B de arriba hacia abajo después de completarla.

## Guía rápida de colaboración y ejecución en Colab

1. Abrir el `.ipynb` con **Google Colab**. Configurar **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Python 3**. CPU es suficiente como punto de partida; no forzar GPU. Instalar dependencias si faltan (por ejemplo: `catboost`, `xgboost`, `lightgbm`, `imbalanced-learn`).
2. Disponer de **`train.csv`**, **`test.csv`** y **`sample_submission.csv`** del ejercicio Santander. Estos CSV **no están incrustados** en este notebook. Guardarlos en Drive o subirlos a `/content`; indicar la ubicación real en la sección **4.1.1**.
3. No es necesario ejecutar las celdas de la **Parte A** para realizar la Parte B. En particular, la Parte A incluye procesamiento anterior a la división de conjuntos, una función de ROC-AUC que debe corregirse y opciones GPU que pueden no funcionar en Colab actual. Se conserva **como referencia docente**, no como fuente de métricas válidas.
4. **No usar `test.csv` de Kaggle como conjunto de evaluación**: carece de `TARGET`. El holdout se debe separar desde `train.csv` y mantenerse sin remuestreo, para medir el desempeño real.
5. Trabajar solo en las celdas asignadas. Si varias personas editan simultáneamente, acordar cómo consolidarán los cambios para evitar sobrescrituras; **Marco verifica la integración**. No cambiar los nombres de variables de la interfaz compartida sin comunicarlo.
6. Al finalizar, usar **Entorno de ejecución → Ejecutar celdas de la Parte B en orden** y comprobar que los resultados se pueden reproducir desde un entorno limpio. No entregar celdas nuevas vacías ni métricas copiadas de ejecuciones antiguas.

**Roles y alcance**

| Integrante | Bloque principal | Entregables |
|---|---|---|
| **Poma** | 4.1.2–4.1.3 y 4.2.1–4.2.4 | Datos, división sin fugas, preprocesamiento y cuatro escaladores |
| **Aldana** | 4.4.1–4.4.5 | Cinco clasificadores nuevos |
| **Brickzon** | 4.5.1–4.5.3 | Tres híbridos VotingClassifier |
| **Vera** | 4.5.4–4.5.5 y 4.6.2 | Dos híbridos StackingClassifier y verificación de ensambles |
| **Marco** | 4.1.1, 4.1.4, 4.3.1 y 4.6.1, 4.6.3–4.6.5 | Entorno Colab, función de evaluación, seis modelos base, comparación y entrega |

**Nota académica:** KNN es un **clasificador** (y también puede emplearse para imputación en otros contextos); no debe contarse como un escalador. Para cumplir el requisito de cuatro transformaciones se usarán cuatro `Scaler/Transformer` de `sklearn.preprocessing`.

# Parte A — Notebook Santander recibido del docente

Las celdas que siguen corresponden al archivo adjunto **`Santander Customer Satisfaction.ipynb`**; se mantuvo su contenido para consulta, retirando únicamente resultados/ejecuciones previas. **No modificar estas celdas como parte de las tareas nuevas.** Se incluye además el ejercicio tal como figura al final. La versión con estratificación y los notebooks de Wine Quality sirven como **referencias externas** y no se fusionan automáticamente aquí.

**Importante:** la función original `Models()` calcula ROC sobre etiquetas predichas en vez de puntuaciones, y el procesamiento original antecede a la división de entrenamiento/prueba. La implementación nueva debe resolver ambos problemas en la **Parte B**, no reutilizar los resultados originales sin corrección.

## Importar librerias

Desde equipos de soporte de primera línea hasta C-suites, la satisfacción del cliente es una medida clave del éxito. Los clientes insatisfechos no se quedan. Además, los clientes descontentos rara vez expresan su insatisfacción antes de irse.

El objetivo es identificar a los clientes insatisfechos en una etapa temprana de su relación. Hacerlo le permitiría a Santander tomar medidas proactivas para mejorar la felicidad de un cliente antes de que sea demasiado tarde.

En esta competencia, trabajará con cientos de funciones anónimas para predecir si un cliente está satisfecho o insatisfecho con su experiencia bancaria.

Se le proporciona un conjunto de datos anonimizados que contiene una gran cantidad de variables numéricas. La columna "OBJETIVO" es la variable a predecir. Es igual a uno para clientes insatisfechos y 0 para clientes satisfechos.

La tarea es predecir la probabilidad de que cada cliente en el conjunto de prueba sea un cliente insatisfecho.
Descripciones de archivos

1. train.csv: el conjunto de entrenamiento que incluye el objetivo  --- 76020
2. test.csv: el conjunto de prueba sin el objetivo  --- 75818
3. sample_submission.csv: un archivo de envío de muestra en el formato correcto

In [ ]:
#generales
import numpy as np
import pandas as pd

#gráficas
import seaborn as sns
import matplotlib.pyplot as plt

#procesamiento
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, RandomizedSearchCV

#modelo
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB

#evaluación
from scipy import stats
from sklearn.metrics import accuracy_score, roc_curve, auc, mean_squared_error, f1_score

import warnings
warnings.filterwarnings('ignore')

* Si no cuenta con alguna de las librerias ejecute el seguiente codigo según corresponda

**Instalación mencionada en la plantilla original (referencia, no ejecutar literalmente):**

```bash
pip install catboost
pip install xgboost
pip install lightgbm
pip install seaborn
pip install scipy
```

## Carga de datos

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

## 1. Análisis Exploratorio

**1.1. muestreo aleatorio del conjunto de datos**

In [ ]:
train.sample(5)

In [ ]:
test.sample(5)

In [ ]:
sample.sample(5)

**1.2. Dimención de la estructura**

In [ ]:
#determinar la dimensionalidad de los conjuntos de datos
# shape (instancias, concepto)
print("Shape de train dataframe : {}".format(train.shape))
print("Shape de test dataframe : {}".format(test.shape))
print("Shape de sample dataframe : {}".format(sample.shape))

**1.3. Descripción y estadística basica**

In [ ]:
#Comprobar la existencia de null's en los conjuntos de datos
print('Null en train : {}'.format(train.isnull().sum().any()))
print('Null en test : {}'.format(test.isnull().sum().any()))

In [ ]:
#análisis estadistico básico de las columnas numericas
train.describe()

In [ ]:
#tipos de datos
train.info()

In [ ]:
#tipos de datos
test.info()

In [ ]:
# Determinar la existencia de duplicados
dupli = train[train.duplicated()]
dupli

**1.4. Análisis Gráfico**

In [ ]:
plt.figure(figsize = (20,20))
for i in range (50):
    plt.subplot(5, 10, i+1)
    sns.boxplot(train.iloc[:,i])
    plt.xlabel(train.columns[i], size = 10)

### 2. Modificación

In [ ]:
# remover columnas con valores unicos
for i in train.columns:
    if train[i].nunique() == 1:
        print(i)
        train.drop(i, inplace = True, axis = 1)
        test.drop(i, inplace = True, axis = 1)

In [ ]:
def Outliers(data, ft):
    IQ1 = data[ft].quantile(0.25)
    IQ3 = data[ft].quantile(0.75)
    IQR = IQ3 - IQ1
    
    lower_bound = IQ1 - 1.5 * IQR
    upper_bound = IQ3 + 1.5 * IQR
    
    return lower_bound, upper_bound

In [ ]:
def replace_outliers_with_iqr(train):
    # Recorremos cada columna del DataFrame
    
    for i in train.columns:
        if i != 'TARGET':
            # Solo aplicar a columnas numéricas
            if train[i].dtype in ['int64', 'float64']:
                # Obtener los límites de los outliers
                lower_bound, upper_bound = Outliers(train, i)
                
                # Reemplazar valores que están por debajo del límite inferior
                train.loc[train[i] < lower_bound, i] = lower_bound
                
                # Reemplazar valores que están por encima del límite superior
                train.loc[train[i] > upper_bound, i] = upper_bound
            
    return train

In [ ]:
train = replace_outliers_with_iqr(train)
train

In [ ]:
print(train['TARGET'].value_counts())
plt.figure(figsize = (8,5))
sns.countplot(x = train['TARGET'])
plt.xlabel('Target', size = 12)
plt.ylabel('Count', size = 12)
plt.title('Distribución en la columna de target antes del remuestreo', size = 12)

In [ ]:
#eliminación de columnas 
test_id = test['ID']
train.drop('ID', inplace = True, axis = 1)
test.drop('ID', inplace = True, axis = 1)

**Separación de variables entrada y target**

In [ ]:
x = train.drop('TARGET', axis = 1)
x.head()

In [ ]:
y = train.loc[:, 'TARGET']
y.head()

**Técnicas de muestreo**

In [ ]:
# Muestreo de los datos, para equilibrar las clases en el target.
sampler = RandomUnderSampler(random_state=123, sampling_strategy=.5)
x, y = sampler.fit_resample(x, y)

In [ ]:
print(y.value_counts())
plt.figure(figsize = (8,5))
sns.countplot(x = y)
plt.xlabel('Target', size = 12)
plt.ylabel('Count', size = 12)
plt.title('Distribución en la columna de target despues del remuestreo', size = 12)

In [ ]:
# aplicar técnicas de escalado
# máximos minimos y standar

col = x.columns
std = StandardScaler()
x_std = std.fit_transform(x)
x_std = pd.DataFrame(data = x_std, columns = col)

mms = MinMaxScaler()
x_mms = mms.fit_transform(x)
x_mms = pd.DataFrame(data = x_mms, columns = col)

### 3. test y entrenamiento

In [ ]:
x_mms.head()

**Creación de clase model**

In [ ]:
def Models(model, name, d, xtrain, ytrain, xtest, ytest):
    print("Working on {} model".format(name))
    
    cla = model
    cla.fit(xtrain, ytrain)
    
    predicted = cla.predict(xtrain)
    tr_auc = accuracy_score(predicted, ytrain)*100
    
    predicted = cla.predict(xtest)
    te_auc = accuracy_score(predicted, ytest)*100
    
    F1 = f1_score(predicted, ytest)
    fpr, tpr, threshould = roc_curve(predicted, ytest)
    AUC = auc(fpr, tpr)
    
    d['Name'].append(name)
    d['Training ACU'].append(tr_auc)
    d['Testing ACU'].append(te_auc)
    d['F1'].append(F1)
    d['AUC'].append(AUC)
    
    print("**********"*5)
    print()
    return d, cla

**Evaluación de modelos segú el conjunto**

In [ ]:
l = [[x, 'Normal'], [x_std, 'Standard'], [x_mms, 'Min Max']]
final = []
model_fit = []
for i in l:
    xtrain, xtest, ytrain, ytest = train_test_split(i[0], y, test_size = 0.25, random_state = 42, stratify = y)
    
    d = {'Name' : [], 'Training ACU': [], 'Testing ACU': [], 'F1': [], 'AUC': []}
    
    models = [ 
        [RandomForestClassifier(n_estimators = 200), 'Random Forest'], 
        [DecisionTreeClassifier(), 'Decision Tree'], 
        [XGBClassifier(tree_method='gpu_hist'), 'XGBoost'], 
        [CatBoostClassifier(task_type="GPU"), 'CatBoost'],
        [GaussianNB(), 'Naive Bayes'], 
        [LogisticRegression(), 'Logistic Regression'] 
    ]

    for model in models:
        d, m = Models(model[0], model[1], d, xtrain, ytrain, xtest, ytest)
        model_fit.append([m, model[1], i[1]])
    final.append(d)
    

**Evaluación de los resultados por modelo y tipo de conjunto de datos**

In [ ]:
name = ['Normal', 'Standard', 'Min Max']
for i in range (len(name)):
    print(name[i])
    acu_data = pd.DataFrame(data = final[i])
    print(acu_data)
    print("******"*12)

# EJERCICIO

* Emplear cuatro modelos de escalamiento; por ejemplo:
    - robust Scaler
    - Basado en modelos (knn, etc.)

* agregar cinco modelos; por ejemplo:
    - SVM
    - AdaBoost

* preparar cinco modelos hibridos en base a los modelos previamente vistos; por ejemplo aplicando la libreria:
    - VotingClassifier

# Parte B — Desarrollo grupal: IMPLEMENTACIÓN PENDIENTE

**Cómo completar esta parte:** cada apartado incluye **(1) título, (2) celda independiente del responsable, (3) indicaciones y criterios de verificación, (4) celda de código vacía**. Si se necesita más de una celda de código, el responsable puede añadirlas debajo de la asignada. Al integrar, conservar esta estructura y escribir conclusiones en nuevas celdas Markdown.

**Contrato técnico común (no cambiar nombres sin acordarlo):**

| Nombre | Tipo y finalidad |
|---|---|
| `X_train`, `X_test` | `pandas.DataFrame` de características, con las mismas columnas y orden |
| `y_train`, `y_test` | `pandas.Series` binaria con `TARGET`: 1 = insatisfecho, 0 = satisfecho |
| `escaladores` | `dict[str, transformer]` con 4 escaladores **sin ajustar** |
| `modelos_originales` | `dict[str, classifier]` con los 6 modelos base |
| `modelos_nuevos` | `dict[str, classifier]` con los 5 modelos nuevos |
| `modelos_hibridos` | `dict[str, classifier]` con H1–H5 |
| `evaluar_modelo` | función común definida por Marco para entrenar/evaluar con el protocolo elegido |
| `resultados` | `list[dict]` o tabla consolidable, con una fila por configuración evaluada |

**Protocolo obligatorio:** `random_state=42` cuando el estimador lo permita; división **75 % entrenamiento / 25 % holdout**, estratificada; ajustar transformaciones y remuestreo **solo sobre entrenamiento**. Elegir hiperparámetros/escaladores mediante validación interna o CV sobre entrenamiento; evaluar el holdout final sin utilizarlo para decidir el modelo. Reportar mínimo Accuracy, Precision, Recall, F1 y ROC-AUC de clase 1. No remuestrear el holdout.

## 4.1. Preparación e interfaz compartida

Este bloque crea los objetos que usarán todas las secciones. Poma y Marco deben cerrar el contrato de nombres antes de que Aldana, Brickzon y Vera empiecen a integrar. Todas las celdas de implementación siguen vacías.

### 4.1.1. Preparar entorno Colab y localizar archivos

> **Responsable: Marco**

**Implementar:** instalar/importar las bibliotecas necesarias en Colab; elegir un método de acceso a datos (Google Drive o subida manual); declarar `DATA_DIR` con una ruta **real**; comprobar si existen `train.csv`, `test.csv` y `sample_submission.csv` antes de leerlos. No incrustar credenciales.

**Sugerencias de comandos (solo como guía):** `%pip install -q catboost xgboost lightgbm imbalanced-learn`. Para Drive: `from google.colab import drive` y `drive.mount('/content/drive')`. Para subida puntual: `from google.colab import files` y `files.upload()`.

**Salida esperada:** mostrar las versiones de Python y `scikit-learn`, la ruta seleccionada y la existencia de los tres CSV. Si faltan, informar claramente cuál falta.

**Verificación:** ejecutar sin rutas de Windows ni referencias a directorios locales ajenos al entorno Colab.

### 4.1.2. Cargar el dataset y construir la división estratificada

> **Responsable: Poma**

**Implementar:** recargar **desde los archivos originales**, sin reutilizar `train`, `x` ni `y` manipulados en la Parte A. Separar `TARGET`, retirar `ID` de los predictores, conservar `ID` aparte si luego se generan predicciones para Kaggle. Construir `X_train`, `X_test`, `y_train`, `y_test` con `train_test_split(test_size=0.25, stratify=y, random_state=42)`.

**Reglas:** `test.csv` carece de `TARGET`; **no** usarlo para métricas. No aplicar ADASYN, RandomUnderSampler, SMOTE, escaladores, clipping de valores atípicos ni selección basada en estadísticas del dataset completo **antes del split**.

**Salida esperada:** dimensiones, lista de características numéricas/no numéricas, distribución de `TARGET` antes y después del split, y confirmación de que los conjuntos no comparten registros.

**Verificación:** mismas columnas y orden para `X_train`/`X_test`; valores binarios esperados 0 y 1, proporciones estratificadas similares.

In [ ]:
# 4.1.2 Carga limpia de train.csv y holdout estratificado
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# No se reutilizan los objetos modificados en la Parte A. DATA_DIR es opcional;
# si no existe, se buscan los CSV subidos en /content o en el directorio actual.
if "DATA_DIR" in globals():
    directorios_datos = [Path(DATA_DIR)]
else:
    directorios_datos = []
directorios_datos += [Path("/content"), Path.cwd(), Path.cwd() / "Semana 13" / "santander_data"]
directorio_csv = next(
    (directorio for directorio in directorios_datos
     if (directorio / "train.csv").is_file()),
    None,
)
if directorio_csv is None:
    revisados = ", ".join(str(ruta) for ruta in directorios_datos)
    raise FileNotFoundError(
        "No se encontró train.csv. Sube train.csv a /content o define DATA_DIR. "
        f"Directorios revisados: {revisados}"
    )

ruta_train = directorio_csv / "train.csv"
train_original = pd.read_csv(ruta_train, low_memory=False)
columnas_requeridas = {"ID", "TARGET"}
if not columnas_requeridas.issubset(train_original.columns):
    raise ValueError(f"train.csv debe contener ID y TARGET; faltan: {columnas_requeridas - set(train_original.columns)}")
if train_original["TARGET"].isna().any():
    raise ValueError("TARGET contiene valores ausentes.")
clases_target = set(train_original["TARGET"].unique())
if clases_target != {0, 1}:
    raise ValueError(f"Se esperaban etiquetas binarias 0 y 1 en TARGET; se encontraron: {clases_target}")

# ID se preserva para trazabilidad, pero no se utiliza como predictor.
ID = train_original["ID"].copy()
if ID.isna().any():
    raise ValueError(f"ID contiene {int(ID.isna().sum())} valores ausentes; no se puede verificar unicidad.")
ids_duplicados = ID[ID.duplicated(keep=False)]
if not ids_duplicados.empty:
    ejemplos_ids = ids_duplicados.drop_duplicates().head(10).tolist()
    raise ValueError(
        f"ID debe ser único antes de dividir; hay {int(ID.duplicated().sum())} filas con ID repetido. "
        f"Ejemplos: {ejemplos_ids}"
    )
y = train_original["TARGET"].astype("int8").copy()
X = train_original.drop(columns=["ID", "TARGET"]).copy()
columnas_no_numericas = X.select_dtypes(exclude="number").columns.tolist()
if columnas_no_numericas:
    raise TypeError(f"Se esperaban predictores numéricos; revisar: {columnas_no_numericas}")

X_train, X_test, y_train, y_test, ID_train, ID_holdout = train_test_split(
    X, y, ID, test_size=0.25, stratify=y, random_state=42
)
if not X_train.index.intersection(X_test.index).empty:
    raise AssertionError("Hay registros compartidos entre entrenamiento y holdout.")
if list(X_train.columns) != list(X_test.columns):
    raise AssertionError("El orden de columnas no coincide entre entrenamiento y holdout.")

print(f"Fuente: {ruta_train}")
print(f"Train original: {train_original.shape}; X: {X.shape}; predictores numéricos: {X.shape[1]}")
print("Columnas identificadoras excluidas de X: ['ID']; TARGET se conserva aparte como y.")
print(f"Particiones: X_train={X_train.shape}, X_test={X_test.shape}")
print("Proporción de TARGET (train original):")
print(y.value_counts(normalize=True).sort_index().rename("proporción").to_string())
print("Conteo y proporción por partición (TARGET=0/1):")
for nombre, etiquetas in [("train", y_train), ("holdout", y_test)]:
    print(f"{nombre}: conteos={etiquetas.value_counts().sort_index().to_dict()}, "
          f"proporciones={etiquetas.value_counts(normalize=True).sort_index().round(5).to_dict()}")
print("ID_train e ID_holdout se preservan aparte; no hay filas compartidas.")
print("Valores ausentes en X_train / X_test:", int(X_train.isna().sum().sum()), "/", int(X_test.isna().sum().sum()))


### 4.1.3. Definir preprocesamiento reproducible y libre de fuga de datos

> **Responsable: Poma**

**Implementar:** proponer un `Pipeline` o una función de construcción de pipelines que ajuste **en entrenamiento** la imputación de faltantes, el escalamiento seleccionado y el clasificador. Si se eliminan columnas constantes o se limitan outliers, aprender esas reglas solo con entrenamiento y aplicar la misma transformación a holdout. Elegir y documentar una estrategia uniforme de desbalanceo (por defecto: sin remuestreo; si usan muestreo, hacerlo con un `imblearn.pipeline.Pipeline` dentro del entrenamiento y de cada fold).

**No hacer:** transformar `X_test` con un `fit_transform`, ni aplicar undersampling/SMOTE a los datos de prueba. No usar `ID` como predictor.

**Salida esperada:** objeto/fábrica reutilizable en las secciones 4.3–4.5; una breve nota de las transformaciones y su orden.

**Verificación:** `fit` ocurre solo sobre muestras de entrenamiento; transformaciones idénticas en las demás particiones.

In [ ]:
# 4.1.3 Fábrica de preprocesamiento sin fuga de datos
import warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

class EliminarColumnasVacias(BaseEstimator, TransformerMixin):
    """Detecta en fit y retira columnas completamente vacías, con aviso explícito."""
    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
            vacias = X.isna().all(axis=0).to_numpy()
        else:
            matriz = np.asarray(X, dtype=float)
            self.feature_names_in_ = np.asarray([f"x{i}" for i in range(matriz.shape[1])], dtype=object)
            vacias = np.isnan(matriz).all(axis=0)
        self.columnas_vacias_ = self.feature_names_in_[vacias].tolist()
        self.columnas_con_datos_ = self.feature_names_in_[~vacias].tolist()
        self.mascara_columnas_ = ~vacias
        if not self.mascara_columnas_.any():
            raise ValueError("Todas las columnas están vacías en el conjunto usado para fit.")
        if self.columnas_vacias_:
            warnings.warn(
                "Se excluyen columnas completamente vacías, detectadas solo en el entrenamiento: "
                f"{self.columnas_vacias_}. La misma selección se aplicará a validación/holdout.",
                UserWarning,
            )
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            faltantes = set(self.feature_names_in_) - set(X.columns)
            if faltantes:
                raise ValueError(f"Faltan columnas vistas durante fit: {sorted(faltantes)}")
            return X.loc[:, self.columnas_con_datos_]
        matriz = np.asarray(X, dtype=float)
        return matriz[:, self.mascara_columnas_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columnas_con_datos_, dtype=object)

# El orden se conserva: detectar columnas vacías en train → imputar mediana
# aprendida en train → filtrar constantes → escalar → clasificar. No se remuestrea.
def crear_pipeline_santander(clasificador, escalador=None):
    """Crea un pipeline sin ajustar; fit debe llamarse solo con X_train/y_train."""
    pasos = [
        ("eliminar_vacias", EliminarColumnasVacias()),
        ("imputador", SimpleImputer(strategy="median")),
        ("selector_constantes", VarianceThreshold(threshold=0.0)),
    ]
    if escalador is not None:
        pasos.append(("escalador", clone(escalador)))
    pasos.append(("clasificador", clone(clasificador)))
    return Pipeline(steps=pasos)

# Para mostrar escaladores sin entrenar modelos, se usa la misma secuencia
# de preprocesamiento de arriba, ajustada exclusivamente sobre X_train.
def demostrar_escalador_en_train(escalador, columnas=None):
    if columnas is None:
        candidatas = X_train.select_dtypes(include="number").columns.tolist()
    else:
        candidatas = list(columnas)
    # Se eligen columnas con al menos dos valores observados distintos en train;
    # así la imputación de mediana no puede dejar una entrada constante al selector.
    columnas = [
        columna for columna in candidatas
        if X_train[columna].nunique(dropna=True) > 1
    ][:8]
    if not columnas:
        raise ValueError("No hay columnas numéricas con variación en X_train para la demostración.")
    preprocesamiento_demo = Pipeline([
        ("eliminar_vacias", EliminarColumnasVacias()),
        ("imputador", SimpleImputer(strategy="median")),
        ("selector_constantes", VarianceThreshold(threshold=0.0)),
    ])
    train_preparado = preprocesamiento_demo.fit_transform(X_train[columnas])
    columnas_preparadas = preprocesamiento_demo.named_steps["selector_constantes"].get_feature_names_out(
        preprocesamiento_demo.named_steps["imputador"].get_feature_names_out(
            preprocesamiento_demo.named_steps["eliminar_vacias"].get_feature_names_out()
        )
    )
    transformador_demo = clone(escalador)
    train_escalado = transformador_demo.fit_transform(train_preparado)
    return train_escalado, list(columnas_preparadas)

print("Pipeline definido: detectar vacías → imputar mediana → filtrar constantes → escalador opcional → clasificador.")
print("El detector informa explícitamente columnas totalmente vacías y ajusta su selección solo con train.")
print("Sin remuestreo: toda transformación aprende al ejecutar fit con X_train/y_train.")
print("Para holdout, usar pipeline.predict(X_test); nunca ajustar ni transformar con fit sobre X_test.")


### 4.1.4. Definir la interfaz de evaluación y resultados

> **Responsable: Marco**

**Implementar:** una función `evaluar_modelo(...)` (o interfaz equivalente, documentando argumentos) que reciba nombre, tipo (`original`, `nuevo` o `hibrido`), clasificador y escalador; construya el pipeline adecuado, entrene en `X_train/y_train` y calcule métricas sobre validación/holdout según el protocolo. Registrar tiempo de entrenamiento y resultados en una **estructura común**, sin cambiar su forma entre integrantes.

**Columnas obligatorias de la tabla final:** `tipo`, `modelo`, `escalador`, `accuracy`, `precision`, `recall`, `f1`, `roc_auc`, `tiempo_s`. Agregar `etapa` (`validacion`/`holdout`) y `semilla` si conviene.

**ROC-AUC correcto:** usar `roc_auc_score(y_real, proba_clase_1)` o `decision_function`; `roc_curve(y_real, puntaje_clase_1)` para curvas ROC. **No** usar `roc_curve(y_pred, y_real)` como en la función original. Controlar divisiones por cero al calcular Precision/Recall/F1. Nunca decidir el mejor modelo basándose reiteradamente en el holdout.

**Validación:** prueba de la interfaz con un clasificador sencillo sobre una partición de validación interna sin modificar el test final; conservar las salidas compatibles con los tres diccionarios de modelos.

## 4.2. Cuatro métodos de escalamiento

Crear el diccionario `escaladores` con **exactamente** estas cuatro entradas: `standard`, `minmax`, `robust`, `quantile`. **No ajustar** los transformadores al crearlos: dejarlos listos para integrarse en los pipelines y ajustar con `fit` exclusivamente sobre el entrenamiento. Comparar qué cambia en distribución/rangos sin usar información del holdout.

### 4.2.1. StandardScaler

> **Responsable: Poma**

**Implementar:** incorporar `StandardScaler` al diccionario `escaladores` con la clave `standard`. Explicar en Markdown la estandarización mediante media y desviación típica.

**Verificación:** comprobar en un subconjunto **de entrenamiento transformado** media aproximadamente 0 y desviación aproximadamente 1 para columnas no constantes; no ajustar al holdout.

**Entregable:** transformador registrable y una interpretación breve.

In [ ]:
# 4.2.1 StandardScaler: centra con la media y escala por desviación estándar
from sklearn.preprocessing import StandardScaler

# Las cuatro demostraciones usan demostrar_escalador_en_train(), que replica
# la imputación, el filtro de constantes y el ajuste únicamente en X_train.
# Las instancias registradas permanecen sin ajustar para la fábrica de pipelines.
escaladores = {"standard": StandardScaler()}
train_standard_demo, columnas_standard_demo = demostrar_escalador_en_train(escaladores["standard"])
print("Escalador registrado (sin ajustar): standard")
print("Demostración independiente, con preprocesamiento compatible sobre X_train.")
print("Columnas variables conservadas:", columnas_standard_demo)
print("Media y desviación poblacional tras estandarizar:")
print("media:", train_standard_demo.mean(axis=0).round(3))
print("desv. estándar:", train_standard_demo.std(axis=0).round(3))


### 4.2.2. MinMaxScaler

> **Responsable: Poma**

**Implementar:** incorporar `MinMaxScaler` con clave `minmax`, rango de entrenamiento `[0, 1]` (a menos que documenten otra elección).

**Verificación:** revisar los mínimos y máximos del **entrenamiento transformado**; valores del holdout pueden quedar fuera de `[0,1]` si exceden el rango aprendido, y eso no debe corregirse con un reajuste.

**Entregable:** transformador y breve interpretación.

In [ ]:
# 4.2.2 MinMaxScaler: transforma usando los mínimos y máximos de entrenamiento
from sklearn.preprocessing import MinMaxScaler

escaladores["minmax"] = MinMaxScaler(feature_range=(0, 1))
train_minmax_demo, columnas_minmax_demo = demostrar_escalador_en_train(escaladores["minmax"])
print("Escalador registrado (sin ajustar): minmax")
print("Demostración independiente, con preprocesamiento compatible sobre X_train.")
print("Mínimos de X_train transformado:", train_minmax_demo.min(axis=0).round(3))
print("Máximos de X_train transformado:", train_minmax_demo.max(axis=0).round(3))
print("Columnas variables conservadas:", columnas_minmax_demo)
print("El holdout puede quedar fuera de [0, 1] si supera los extremos vistos en train; no se reajusta.")


### 4.2.3. RobustScaler

> **Responsable: Poma**

**Implementar:** incorporar `RobustScaler` con clave `robust`; explicar por qué mediana y rango intercuartílico reducen la influencia de valores extremos.

**Verificación:** comparar con StandardScaler usando ejemplos de características con outliers del **entrenamiento**. No eliminar automáticamente valores atípicos sin justificarlo.

**Entregable:** transformador y breve interpretación.

In [ ]:
# 4.2.3 RobustScaler: usa mediana y rango intercuartílico
from sklearn.preprocessing import RobustScaler, StandardScaler

escaladores["robust"] = RobustScaler()
train_robust_demo, columnas_robust_demo = demostrar_escalador_en_train(escaladores["robust"])
train_standard_comparacion, _ = demostrar_escalador_en_train(StandardScaler())
print("Escalador registrado (sin ajustar): robust")
print("Demostración independiente, con preprocesamiento compatible sobre X_train.")
print("Medianas tras RobustScaler:", np.median(train_robust_demo, axis=0).round(3))
print("La mediana y el rango intercuartílico son menos sensibles a valores extremos que media/desviación estándar.")
print("Desviaciones estándar de la misma muestra procesada:")
print("Standard:", train_standard_comparacion.std(axis=0).round(3))
print("Robust:", train_robust_demo.std(axis=0).round(3))
print("Columnas variables conservadas:", columnas_robust_demo)
print("No se eliminan ni recortan outliers en este preprocesamiento.")


### 4.2.4. QuantileTransformer

> **Responsable: Poma**

**Implementar:** incorporar `QuantileTransformer` con clave `quantile`, `output_distribution='normal'`, `random_state=42` y un `n_quantiles` no mayor que el tamaño de entrenamiento usado para ajustar.

**Verificación:** confirmar transformación de distribuciones en una muestra de entrenamiento; explicar que este método puede cambiar distancias/relaciones y por eso debe compararse empíricamente.

**Entregable:** cuarto transformador; mostrar `list(escaladores)` con **exactamente cuatro** claves distintas.

In [ ]:
# 4.2.4 QuantileTransformer: lleva cada marginal a una distribución normal
from sklearn.preprocessing import QuantileTransformer

n_cuantiles = min(1000, len(X_train))
escaladores["quantile"] = QuantileTransformer(
    n_quantiles=n_cuantiles,
    output_distribution="normal",
    random_state=42,
)
train_quantile_demo, columnas_quantile_demo = demostrar_escalador_en_train(escaladores["quantile"])
print("Escalador registrado (sin ajustar): quantile")
print("Demostración independiente, con preprocesamiento compatible sobre X_train.")
print(f"n_quantiles={n_cuantiles}; tamaño de X_train={len(X_train)}")
print("Media de la muestra transformada:", train_quantile_demo.mean(axis=0).round(3))
print("Columnas variables conservadas:", columnas_quantile_demo)
print("El método transforma las distribuciones marginales y puede cambiar distancias/relaciones; su efecto se compara empíricamente.")
print("Escaladores disponibles:", list(escaladores))
if set(escaladores) != {"standard", "minmax", "robust", "quantile"}:
    raise AssertionError("El registro debe contener exactamente los cuatro escaladores acordados.")


## 4.3. Seis clasificadores originales como línea base

### 4.3.1. Reconstruir los seis modelos de la clase bajo el nuevo protocolo

> **Responsable: Marco**

**Implementar:** registrar en `modelos_originales` estos seis clasificadores utilizados en el notebook Santander del docente: `random_forest`, `decision_tree`, `xgboost`, `catboost`, `gaussian_nb`, `logistic_regression`. Usar semillas cuando corresponda y una configuración razonable para Colab.

**Precaución:** el notebook original fuerza GPU en XGBoost/CatBoost y podría fallar en el entorno actual. Para esta plantilla, preferir configuraciones CPU compatibles; solo activar GPU cuando se haya verificado disponibilidad. Evitar copiar la función original `Models()` por su implementación incorrecta de ROC.

**Comparación:** entrenar y evaluar las líneas base con la interfaz común, sin mezclar métricas del notebook original con las de esta Parte B. Para cuantificar el efecto del escalamiento, usar los cuatro transformadores en condiciones comparables; si el costo impide evaluar algunas combinaciones, justificar explícitamente qué se ejecutó y qué no.

**Salida esperada:** `len(modelos_originales) == 6` y resultados trazables con el mismo split/semilla/métricas.

## 4.4. Cinco clasificadores adicionales

Los **cinco modelos de este apartado son nuevos respecto a la lista de seis modelos entrenados en la Parte A**. Registrarlos en `modelos_nuevos`, usando el mismo protocolo de evaluación de 4.1.4. No reescribir el split ni ajustar escaladores sobre todo el dataset. **Aldana** crea el diccionario en 4.4.1 y lo completa en los siguientes apartados.

### 4.4.1. SVM de clasificación (LinearSVC)

> **Responsable: Aldana**

**Implementar:** `LinearSVC` como variante lineal de SVM para controlar el costo computacional del dataset Santander; registrar con clave `svm_lineal`. Usar escalamiento e incrementar iteraciones solo si es necesario, explicando convergencia y margen.

**ROC-AUC:** `LinearSVC` no expone `predict_proba` de forma nativa; usar `decision_function` para ROC-AUC. No incluirlo directamente en híbridos de `voting='soft'` sin calibrar probabilidades.

**Entregable:** entrenamiento, métricas comparables e interpretación del comportamiento de SVM.

### 4.4.2. AdaBoostClassifier

> **Responsable: Aldana**

**Implementar:** `AdaBoostClassifier` como `adaboost`, con `random_state=42`; elegir un número razonable de estimadores y documentarlo.

**Verificación:** registrar Accuracy, Precision, Recall, F1 y ROC-AUC; comparar resultados con Decision Tree, su posible clasificador base.

**Entregable:** modelo registrable, métricas e interpretación.

### 4.4.3. KNeighborsClassifier (KNN)

> **Responsable: Aldana**

**Implementar:** `KNeighborsClassifier` como `knn`. Elegir un valor inicial de `n_neighbors` y, si hay tiempo, compararlo mediante validación **solo sobre entrenamiento**.

**Precaución:** KNN es sensible a la escala, dimensión y costo de distancia; no contarlo como uno de los cuatro escaladores y evitar `KNNImputer` masivo salvo necesidad y justificación.

**Entregable:** modelo registrable y explicación de qué transformador le favoreció.

### 4.4.4. LightGBMClassifier

> **Responsable: Aldana**

**Implementar:** `LGBMClassifier` como `lightgbm`, con semilla fija y parámetros moderados para Colab. Aunque la biblioteca aparezca en las **importaciones** originales, **no figura entre los seis clasificadores entrenados**, así que cuenta como modelo adicional.

**Verificación:** métricas sobre particiones acordadas y comparación con XGBoost/CatBoost.

**Entregable:** modelo registrado, configuración y evaluación.

### 4.4.5. ExtraTreesClassifier

> **Responsable: Aldana**

**Implementar:** `ExtraTreesClassifier` como `extra_trees`; fijar `random_state=42` y documentar número de árboles.

**Verificación:** comparar su comportamiento con Random Forest y explicar el mayor grado de aleatoriedad de Extra Trees.

**Entregable:** modelo registrado, métricas y comprobación final de `len(modelos_nuevos) == 5`.

## 4.5. Cinco modelos híbridos (ensambles)

Cada híbrido debe construirse con estimadores presentes en `modelos_originales` o `modelos_nuevos`. Registrar **H1–H5** en `modelos_hibridos`. En Voting Soft solo usar clasificadores que entregan probabilidades; en Stacking prevenir fugas con CV sobre entrenamiento. Todos se evalúan mediante la interfaz compartida. Los híbridos se registran por separado de los 11 modelos individuales.

### 4.5.1. H1 — VotingClassifier Hard: Logistic Regression + Decision Tree + KNN

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='hard')` con los tres estimadores indicados; registrar clave `H1_voting_hard`. Incorporar preprocesamiento adecuado para KNN/Logistic Regression dentro del procedimiento acordado.

**Explicar:** la etiqueta final se determina por voto mayoritario. **Precaución de evaluación:** en general hard voting no expone probabilidades; **no inventar ROC-AUC**. Si no existe puntuación apropiada, registrar `NaN` para ROC-AUC o evaluar una variante soft aparte documentada sin sustituir H1.

**Entregable:** híbrido entrenable, métricas válidas y breve explicación del mecanismo de voto.

In [ ]:
# 4.5.1 H1 — VotingClassifier Hard: Logistic Regression + Decision Tree + KNN
import numpy as np
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone

if "modelos_hibridos" not in globals():
    modelos_hibridos = {}

# Estimadores base: tomados de los diccionarios o instanciados con configuración estándar
est_lr = (
    clone(modelos_originales["logistic_regression"])
    if "modelos_originales" in globals() and "logistic_regression" in modelos_originales
    else LogisticRegression(max_iter=1000, random_state=42)
)

est_dt = (
    clone(modelos_originales["decision_tree"])
    if "modelos_originales" in globals() and "decision_tree" in modelos_originales
    else DecisionTreeClassifier(random_state=42)
)

est_knn = (
    clone(modelos_nuevos["knn"])
    if "modelos_nuevos" in globals() and "knn" in modelos_nuevos
    else KNeighborsClassifier(n_neighbors=5)
)

# Construcción de H1 con voto mayoritario (Hard Voting)
h1_voting_hard = VotingClassifier(
    estimators=[
        ("logistic_regression", est_lr),
        ("decision_tree", est_dt),
        ("knn", est_knn)
    ],
    voting="hard"
)

modelos_hibridos["H1_voting_hard"] = h1_voting_hard

print("H1 registrado con éxito en modelos_hibridos['H1_voting_hard']:")
print(" - Estimadores: Logistic Regression, Decision Tree, KNN")
print(" - Método: voting='hard' (voto mayoritario sobre clases discretas)")
print(" - Preprocesamiento: KNN y Logistic Regression son sensibles a la escala; el ensamble completo debe evaluarse con un escalador (ej. StandardScaler o RobustScaler) a través del pipeline sin fuga.")
print(" - Precaución métricas: Al no generar probabilidades continuas, no se inventa ROC-AUC; por protocolo se reporta NaN para ROC-AUC.")

# Evaluación automática si la interfaz compartida y los datos ya están disponibles
if "evaluar_modelo" in globals() and callable(evaluar_modelo) and "X_train" in globals():
    try:
        esc = escaladores.get("standard") if "escaladores" in globals() else None
        res_h1 = evaluar_modelo("H1_voting_hard", "hibrido", h1_voting_hard, esc)
        print("Evaluación inicial de H1 completada.")
    except Exception as e:
        print(f"Nota: Evaluación diferida para la integración en 4.6 ({e}).")


### 4.5.2. H2 — VotingClassifier Soft: Random Forest + AdaBoost + Logistic Regression

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='soft')` con los tres estimadores; registrar `H2_voting_soft`.

**Explicar:** combina probabilidades estimadas de clase 1. Comprobar que los tres modelos tengan `predict_proba`; usar sus puntuaciones para ROC-AUC de forma correcta. Escalamiento ajustado solo en entrenamiento.

**Entregable:** híbrido entrenable, métricas y comparación con cada componente.

In [ ]:
# 4.5.2 H2 — VotingClassifier Soft: Random Forest + AdaBoost + Logistic Regression
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

if "modelos_hibridos" not in globals():
    modelos_hibridos = {}

# Estimadores base con soporte para predict_proba
est_rf = (
    clone(modelos_originales["random_forest"])
    if "modelos_originales" in globals() and "random_forest" in modelos_originales
    else RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
)

est_ada = (
    clone(modelos_nuevos["adaboost"])
    if "modelos_nuevos" in globals() and "adaboost" in modelos_nuevos
    else AdaBoostClassifier(random_state=42)
)

est_lr = (
    clone(modelos_originales["logistic_regression"])
    if "modelos_originales" in globals() and "logistic_regression" in modelos_originales
    else LogisticRegression(max_iter=1000, random_state=42)
)

# Construcción de H2 con promedio de probabilidades (Soft Voting)
h2_voting_soft = VotingClassifier(
    estimators=[
        ("random_forest", est_rf),
        ("adaboost", est_ada),
        ("logistic_regression", est_lr)
    ],
    voting="soft"
)

modelos_hibridos["H2_voting_soft"] = h2_voting_soft

print("H2 registrado con éxito en modelos_hibridos['H2_voting_soft']:")
print(" - Estimadores: Random Forest, AdaBoost, Logistic Regression")
print(" - Método: voting='soft' (promedio de probabilidades predichas de clase 1)")
print(" - Verificación predict_proba: Los tres clasificadores disponen de predicción probabilística calibrada para cálculo legítimo de ROC-AUC.")

# Evaluación automática si la interfaz compartida y los datos ya están disponibles
if "evaluar_modelo" in globals() and callable(evaluar_modelo) and "X_train" in globals():
    try:
        esc = escaladores.get("standard") if "escaladores" in globals() else None
        res_h2 = evaluar_modelo("H2_voting_soft", "hibrido", h2_voting_soft, esc)
        print("Evaluación inicial de H2 completada.")
    except Exception as e:
        print(f"Nota: Evaluación diferida para la integración en 4.6 ({e}).")


### 4.5.3. H3 — VotingClassifier Soft: XGBoost + LightGBM + CatBoost

> **Responsable: Brickzon**

**Implementar:** `VotingClassifier(voting='soft')` con XGBoost, LightGBM y CatBoost; registrar `H3_voting_soft_boosting`.

**Precaución Colab:** usar versiones/argumentos compatibles, preferentemente CPU sin forzar GPU ni usar opciones obsoletas. Mantener semilla y criterios de evaluación comunes; usar estimadores clonables.

**Entregable:** híbrido entrenable, métricas y explicación de si combinar tres métodos boosting agrega valor respecto a cada uno.

In [ ]:
# 4.5.3 H3 — VotingClassifier Soft: XGBoost + LightGBM + CatBoost
from sklearn.ensemble import VotingClassifier
from sklearn.base import clone
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

if "modelos_hibridos" not in globals():
    modelos_hibridos = {}

# Estimadores boosting con parámetros seguros para CPU en Colab
est_xgb = (
    clone(modelos_originales["xgboost"])
    if "modelos_originales" in globals() and "xgboost" in modelos_originales
    else XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )
)

est_lgb = (
    clone(modelos_nuevos["lightgbm"])
    if "modelos_nuevos" in globals() and "lightgbm" in modelos_nuevos
    else LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
)

est_cat = (
    clone(modelos_originales["catboost"])
    if "modelos_originales" in globals() and "catboost" in modelos_originales
    else CatBoostClassifier(
        iterations=100,
        learning_rate=0.1,
        depth=5,
        random_seed=42,
        verbose=0,
        thread_count=-1
    )
)

# Construcción de H3 combinando las tres bibliotecas principales de Gradient Boosting
h3_voting_soft_boosting = VotingClassifier(
    estimators=[
        ("xgboost", est_xgb),
        ("lightgbm", est_lgb),
        ("catboost", est_cat)
    ],
    voting="soft"
)

modelos_hibridos["H3_voting_soft_boosting"] = h3_voting_soft_boosting

print("H3 registrado con éxito en modelos_hibridos['H3_voting_soft_boosting']:")
print(" - Estimadores: XGBoost, LightGBM, CatBoost")
print(" - Método: voting='soft' (promedio de probabilidades de boosting)")
print(" - Configuración: CPU multi-hilo, sin opciones obsoletas de GPU, totalmente reproducible (semilla 42).")
print(f"Total de modelos híbridos registrados actualmente: {len(modelos_hibridos)}")

# Evaluación automática si la interfaz compartida y los datos ya están disponibles
if "evaluar_modelo" in globals() and callable(evaluar_modelo) and "X_train" in globals():
    try:
        esc = escaladores.get("standard") if "escaladores" in globals() else None
        res_h3 = evaluar_modelo("H3_voting_soft_boosting", "hibrido", h3_voting_soft_boosting, esc)
        print("Evaluación inicial de H3 completada.")
    except Exception as e:
        print(f"Nota: Evaluación diferida para la integración en 4.6 ({e}).")


### 4.5.4. H4 — StackingClassifier: SVM + KNN + Random Forest

> **Responsable: Vera**

**Implementar:** `StackingClassifier` con `LinearSVC`, `KNeighborsClassifier` y `RandomForestClassifier` como modelos base; `LogisticRegression` como meta-modelo. Registrar `H4_stacking_svm`.

**Condiciones:** emplear `cv=3` (o justificar otro valor), `stack_method='auto'` y preprocesamiento compatible; construir pipelines internos para modelos sensibles a escala cuando corresponda. El meta-modelo se entrena con predicciones obtenidas **mediante CV del entrenamiento**; nunca con etiquetas del holdout.

**Entregable:** híbrido, reporte de costo de entrenamiento, métricas y explicación de las dos capas.

### 4.5.5. H5 — StackingClassifier: Decision Tree + Extra Trees + AdaBoost

> **Responsable: Vera**

**Implementar:** `StackingClassifier` con `DecisionTreeClassifier`, `ExtraTreesClassifier` y `AdaBoostClassifier` como modelos base; `LogisticRegression` como meta-modelo. Registrar `H5_stacking_arboles`.

**Condiciones:** CV interna sobre entrenamiento, semillas consistentes, evitar remuestreo fuera de folds y no modificar datos de validación/holdout. Medir tiempo y observar sobreajuste.

**Entregable:** híbrido, métricas comparables, contraste con H4 y comprobación de `len(modelos_hibridos) == 5`.

## 4.6. Evaluación final, auditoría e integración

### 4.6.1. Consolidar resultados y seleccionar configuraciones mediante validación

> **Responsable: Marco**

**Implementar:** ejecutar los clasificadores `modelos_originales` y `modelos_nuevos` bajo los cuatro escaladores usando la misma estrategia de validación (idealmente CV estratificada en **entrenamiento**). Para los híbridos, seleccionar escalador/preprocesamiento usando la misma validación, registrando la decisión. Consolidar `resultados` en `tabla_resultados` con columnas homogéneas.

**Matriz sugerida:** hasta **4 × 11 = 44 configuraciones individuales** y **5 híbridos**. Esto es una **propuesta de evaluación**, no un número expresamente exigido por el profesor; si el costo de Colab obliga a priorizar, documentar cobertura real y justificarla. Si se usa una muestra exploratoria, las comparaciones deben usar la **misma muestra** y las conclusiones finales deben identificarse como tales.

**Selección:** priorizar ROC-AUC, Recall y F1 de clase 1; Accuracy no debe decidir sola por el desbalanceo. No elegir repetidamente configuraciones mirando el holdout.

**Entregable:** tabla completa, ordenamiento por métrica prioritaria y registro de configuraciones efectivamente probadas.

### 4.6.2. Verificar los cinco híbridos y sus métricas

> **Responsable: Vera**

**Implementar:** comprobar que `modelos_hibridos` contiene H1–H5 y que sus componentes están presentes entre los modelos individuales. Evaluar los cinco con la interfaz común y un preprocesamiento comparable.

**Auditoría mínima:** comparar H1/H2/H3 frente a sus componentes; H4/H5 frente a sus modelos base; revisar `ROC-AUC` con etiquetas reales y puntuaciones (para hard voting, `NaN` si no hay puntuación válida); registrar tiempos y errores/convergencia sin ocultarlos.

**Entregable:** tabla `tabla_hibridos`, con cinco filas por la configuración final elegida y una nota sobre rendimiento/costo.

### 4.6.3. Generar visualizaciones y matrices de confusión

> **Responsable: Marco**

**Implementar:** tabla ordenada de rendimiento individual e híbrido, gráficos legibles (por ejemplo, barras de ROC-AUC/F1/Recall) y matrices de confusión de los modelos más relevantes usando el **holdout final**. Si se trazan curvas ROC, usar predicciones probabilísticas o puntuaciones sobre `X_test` y `y_test`, sin invertir los argumentos.

**Entregable:** gráficos con etiquetas y títulos, interpretación de falsos negativos (clientes insatisfechos no detectados) y advertencia del desbalanceo.

**Verificación:** las etiquetas y los cálculos provienen de la misma partición, no mezclan validación con holdout.

### 4.6.4. Redactar interpretación, conclusiones y limitaciones

> **Responsable: Marco**

**Implementar en celdas de texto Markdown debajo de este bloque:** justificar cuál escalador funcionó mejor y para qué modelos; comparar individuales versus híbridos; explicar qué métricas sostienen la selección; comentar tiempo computacional, desbalanceo y límites de la evaluación.

**No inventar resultados.** Completar las conclusiones únicamente después de ejecutar las celdas y revisar la tabla consolidada. Indicar si alguna combinación se omitió y el motivo.

**Entregable:** un párrafo de resultados, un párrafo comparativo y un párrafo de limitaciones y mejoras. La celda de código vacía se deja como espacio opcional para cálculos auxiliares de este análisis.

### 4.6.5. Control final de integración y ejecución en Google Colab

> **Responsable: Marco**

**Implementar:** comprobaciones automáticas/focalizadas que verifiquen: 4 escaladores, 6 modelos originales, 5 modelos nuevos, 5 híbridos; forma y clases de `X_train/X_test/y_train/y_test`; inexistencia de `ID` entre predictores; ausencia de fuga antes de split; nombres únicos de modelos y métricas válidas. Verificar que `tabla_resultados` no esté vacía y no contenga resultados inventados.

**Entrega:** guardar el `.ipynb` final con Markdown completo, celdas de ejecución y resultados reales, más una copia descargable desde Colab. Si se utilizó Drive, eliminar dependencias de rutas privadas o documentar cómo reconstruirlas. Ejecutar desde una sesión limpia **la Parte B en orden**, sin depender de variables definidas en la Parte A.

**Criterio de aceptación:** cualquier integrante puede abrir el notebook, ubicar los datos, seguir las instrucciones, ejecutar el bloque grupal y entender quién hizo cada parte.

## 4.7. Lista de comprobación del equipo antes de entregar

- [ ] **Marco:** los tres CSV están accesibles en Colab y el protocolo está definido.
- [ ] **Poma:** split antes de ajustes/remuestreo; existen 4 escaladores definidos sin fugas.
- [ ] **Marco:** seis modelos base comparados de forma válida.
- [ ] **Aldana:** cinco modelos nuevos y sus métricas están registrados.
- [ ] **Brickzon:** H1, H2 y H3 se construyen y se evalúan correctamente.
- [ ] **Vera:** H4 y H5 se construyen; cinco híbridos auditados.
- [ ] **Marco:** resultados, gráficos, conclusiones y ejecución limpia de Parte B.
- [ ] **Equipo:** se documentaron limitaciones, versiones y combinaciones omitidas si existieron.

> **Estado de esta plantilla:** planificación y celdas asignadas, **sin implementaciones nuevas ni resultados calculados**.